# Generator Embeddingów dla Poszczególnych Prac

Ten notebook generuje embeddingi dla każdej pracy naukowej osobno.

## Struktura:
- Każda praca ma **dwa embeddingi**: title i abstract
- Każdy embedding zapisywany w osobnym pliku
- Format nazwy: `{openalex_id}_title.pkl` lub `{openalex_id}_abstract.pkl`
- Możliwość wznowienia - pomija już wygenerowane embeddingi

## 1. Setup i Import

In [1]:
# Instalacja bibliotek
!pip install sentence-transformers pandas numpy tqdm -q

In [2]:
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
import pickle
import os
from tqdm.auto import tqdm
from pathlib import Path
from typing import Optional
import warnings
warnings.filterwarnings('ignore')

print("✅ Biblioteki załadowane")

✅ Biblioteki załadowane


## 2. Konfiguracja

In [ ]:
# Ścieżki
WORKS_FILE = '../data/titles_with_abstracts.csv'
EMBEDDINGS_BASE_DIR = '../embeddings'  # Katalog bazowy dla embeddingów

# ========================================
# KONFIGURACJA: MODELE I TYPY EMBEDDINGÓW
# ========================================

# Lista modeli do użycia - możesz dodać dowolne modele
MODELS = [
    'all-MiniLM-L6-v2',
    'allenai/specter',
    # 'all-mpnet-base-v2',
    # 'paraphrase-multilingual-MiniLM-L12-v2',
]

# Typy embeddingów do wygenerowania
# Możliwe wartości: 'title', 'abstract', 'n_grams'
EMB_TYPES = [
    'title',
    'abstract',
    # 'n_grams',  # Embeddingi N-gramów zdań z abstractu
]

# ========================================
# KONFIGURACJA N-GRAMÓW
# ========================================

# Maksymalna długość N-gramu (ile zdań)
# Jeśli None, używa długości całego abstractu
MAX_N_GRAM = None  # None = wszystkie możliwe N-gramy do długości abstractu

# Minimalna długość N-gramu (zazwyczaj 1 = pojedyncze zdania)
MIN_N_GRAM = 1

# ========================================

# Parametry
BATCH_SIZE = 32  # Rozmiar batcha dla embedowania
SKIP_EXISTING = True  # Czy pomijać już wygenerowane embeddingi

# ========================================

# Utwórz strukturę katalogów
os.makedirs(EMBEDDINGS_BASE_DIR, exist_ok=True)

print(f"📋 Konfiguracja:")
print(f"   Modele: {MODELS}")
print(f"   Typy embeddingów: {EMB_TYPES}")
if 'n_grams' in EMB_TYPES:
    print(f"   N-gramy: min={MIN_N_GRAM}, max={MAX_N_GRAM if MAX_N_GRAM else 'auto'}")
print(f"   Plik danych: {WORKS_FILE}")
print(f"   Katalog bazowy: {EMBEDDINGS_BASE_DIR}")
print(f"   Batch size: {BATCH_SIZE}")
print(f"   Pomijaj istniejące: {SKIP_EXISTING}")
print()
print(f"📊 Do wygenerowania:")
print(f"   {len(MODELS)} modeli × {len(EMB_TYPES)} typów = {len(MODELS) * len(EMB_TYPES)} kombinacji")

## 3. Wczytanie danych

In [5]:
print("📂 Wczytywanie danych...")

df_works = pd.read_csv(WORKS_FILE)

print(f"✅ Załadowano {len(df_works)} prac")
print(f"\n📊 Dostępne kolumny:")
print(f"   {df_works.columns.tolist()}")

# Sprawdź wymagane kolumny
required_cols = ['openalex_id', 'title', 'abstract']
missing_cols = [col for col in required_cols if col not in df_works.columns]
if missing_cols:
    raise ValueError(f"Brak wymaganych kolumn: {missing_cols}")

# Analiza brakujących danych PRZED filtracją
print(f"\n🔍 ANALIZA BRAKUJĄCYCH DANYCH:")
print(f"   Prace bez openalex_id: {df_works['openalex_id'].isna().sum()}")
print(f"   Prace bez title: {df_works['title'].isna().sum()}")
print(f"   Prace bez abstract: {df_works['abstract'].isna().sum()}")

# Policz ile prac zostanie odrzuconych
missing_any = df_works[required_cols].isna().any(axis=1).sum()
print(f"   Prace z brakującym którymikolwiek z pól: {missing_any}")

# Filtruj tylko prace z wypełnionymi polami
df_works_clean = df_works.dropna(subset=['openalex_id', 'title', 'abstract']).copy()

print(f"\n🧹 PO FILTRACJI:")
print(f"   Prace do przetworzenia: {len(df_works_clean)}")
print(f"   Odrzucono: {len(df_works) - len(df_works_clean)} prac")
print(f"   Procent zachowanych: {len(df_works_clean)/len(df_works)*100:.1f}%")

📂 Wczytywanie danych...


FileNotFoundError: [Errno 2] No such file or directory: '../data/titles_with_abstracts.csv'

## 4. Funkcje pomocnicze

In [ ]:
import re

def split_into_sentences(text: str) -> list:
    """
    Dzieli tekst na zdania.
    
    Args:
        text: Tekst do podziału
    
    Returns:
        Lista zdań
    """
    # Prosty podział po znakach kończących zdanie
    # Uwzględnia . ! ? z możliwym białym znakiem po nich
    sentences = re.split(r'(?<=[.!?])\s+', text.strip())
    
    # Usuń puste zdania i oczyść
    sentences = [s.strip() for s in sentences if s.strip()]
    
    return sentences


def create_sentence_ngrams(sentences: list, min_n: int = 1, max_n: int = None) -> list:
    """
    Tworzy N-gramy ze zdań.
    
    Args:
        sentences: Lista zdań
        min_n: Minimalna długość N-gramu
        max_n: Maksymalna długość N-gramu (None = długość całej listy)
    
    Returns:
        Lista N-gramów (każdy N-gram to string złożony ze zdań)
    """
    if not sentences:
        return []
    
    if max_n is None:
        max_n = len(sentences)
    else:
        max_n = min(max_n, len(sentences))
    
    ngrams = []
    
    # Dla każdej długości N-gramu
    for n in range(min_n, max_n + 1):
        # Dla każdej możliwej pozycji startowej
        for i in range(len(sentences) - n + 1):
            # Pobierz N kolejnych zdań
            ngram_sentences = sentences[i:i+n]
            # Połącz je w jeden tekst
            ngram_text = ' '.join(ngram_sentences)
            ngrams.append(ngram_text)
    
    return ngrams


def save_work_embedding(
    embedding: np.ndarray,
    openalex_id: str,
    emb_type: str,
    model_name: str,
    text: str,
    output_dir: str,
    metadata: dict = None
) -> str:
    """
    Zapisuje embedding pracy do pliku.
    
    Args:
        embedding: Wygenerowany embedding (numpy array)
        openalex_id: ID pracy w OpenAlex
        emb_type: Typ embedowania ('title', 'abstract', 'n_grams')
        model_name: Nazwa użytego modelu
        text: Oryginalny tekst (dla referencji)
        output_dir: Katalog docelowy
        metadata: Dodatkowe metadane (opcjonalne)
    
    Returns:
        Ścieżka do zapisanego pliku
    """
    filename = f"{openalex_id}_{emb_type}.pkl"
    filepath = os.path.join(output_dir, filename)
    
    data = {
        'embedding': embedding,
        'openalex_id': openalex_id,
        'type': emb_type,
        'model': model_name,
        'text': text,
        'embedding_dim': len(embedding)
    }
    
    # Dodaj opcjonalne metadane
    if metadata:
        data['metadata'] = metadata
    
    with open(filepath, 'wb') as f:
        pickle.dump(data, f)
    
    return filepath


def load_work_embedding(openalex_id: str, emb_type: str, input_dir: str) -> Optional[dict]:
    """
    Wczytuje embedding pracy z pliku.
    
    Args:
        openalex_id: ID pracy w OpenAlex
        emb_type: Typ embedowania ('title', 'abstract', 'n_grams')
        input_dir: Katalog źródłowy
    
    Returns:
        Dict z danymi lub None jeśli plik nie istnieje
    """
    filename = f"{openalex_id}_{emb_type}.pkl"
    filepath = os.path.join(input_dir, filename)
    
    if not os.path.exists(filepath):
        return None
    
    with open(filepath, 'rb') as f:
        data = pickle.load(f)
    
    return data


def embedding_exists(openalex_id: str, emb_type: str, output_dir: str) -> bool:
    """
    Sprawdza czy embedding już istnieje.
    
    Args:
        openalex_id: ID pracy w OpenAlex
        emb_type: Typ embedowania ('title', 'abstract', 'n_grams')
        output_dir: Katalog do sprawdzenia
    
    Returns:
        True jeśli plik istnieje, False w przeciwnym razie
    """
    filename = f"{openalex_id}_{emb_type}.pkl"
    filepath = os.path.join(output_dir, filename)
    return os.path.exists(filepath)


def generate_ngram_embedding_with_centroid(
    text: str,
    model: SentenceTransformer,
    min_n: int = 1,
    max_n: int = None
) -> tuple:
    """
    Generuje embedding N-gramowy z centrum (centroid).
    
    Proces:
    1. Dzieli tekst na zdania
    2. Tworzy N-gramy zdań (1-gram, 2-gram, ..., N-gram)
    3. Embeduje każdy N-gram
    4. Oblicza centrum (średnią) wszystkich embeddingów
    
    Args:
        text: Tekst do zaembedowania (abstract)
        model: Model do embedowania
        min_n: Minimalna długość N-gramu
        max_n: Maksymalna długość N-gramu
    
    Returns:
        Tuple (centroid_embedding, metadata_dict)
        metadata zawiera: num_sentences, num_ngrams, ngrams_lengths
    """
    # Podziel na zdania
    sentences = split_into_sentences(text)
    
    if not sentences:
        return None, None
    
    # Utwórz N-gramy
    ngrams = create_sentence_ngrams(sentences, min_n=min_n, max_n=max_n)
    
    if not ngrams:
        return None, None
    
    # Embeduj wszystkie N-gramy
    ngram_embeddings = model.encode(
        ngrams,
        batch_size=len(ngrams),
        show_progress_bar=False,
        convert_to_numpy=True
    )
    
    # Oblicz centrum (centroid) - średnia wszystkich embeddingów
    centroid = np.mean(ngram_embeddings, axis=0)
    
    # Przygotuj metadane
    metadata = {
        'num_sentences': len(sentences),
        'num_ngrams': len(ngrams),
        'ngram_lengths': [n for n in range(min_n, min(max_n or len(sentences), len(sentences)) + 1)],
        'min_n': min_n,
        'max_n': max_n
    }
    
    return centroid, metadata


def generate_embeddings_batch(
    df_batch: pd.DataFrame,
    model: SentenceTransformer,
    text_column: str,
    emb_type: str,
    output_dir: str,
    model_name: str,
    skip_existing: bool = True,
    min_n_gram: int = 1,
    max_n_gram: int = None
) -> dict:
    """
    Generuje embeddingi dla batcha prac.
    
    Args:
        df_batch: DataFrame z pracami
        model: Załadowany model
        text_column: Kolumna z tekstem ('title', 'abstract')
        emb_type: Typ embedowania ('title', 'abstract', 'n_grams')
        output_dir: Katalog docelowy
        model_name: Nazwa modelu
        skip_existing: Czy pomijać istniejące
        min_n_gram: Minimalna długość N-gramu (dla n_grams)
        max_n_gram: Maksymalna długość N-gramu (dla n_grams)
    
    Returns:
        Dict ze statystykami: {'generated': int, 'skipped': int, 'failed': int}
    """
    stats = {'generated': 0, 'skipped': 0, 'failed': 0}
    
    # Dla N-gramów przetwarzamy każdą pracę osobno
    if emb_type == 'n_grams':
        for idx, row in df_batch.iterrows():
            openalex_id = row['openalex_id']
            
            # Sprawdź czy już istnieje
            if skip_existing and embedding_exists(openalex_id, emb_type, output_dir):
                stats['skipped'] += 1
                continue
            
            try:
                text = str(row[text_column])
                
                # Generuj embedding N-gramowy z centrum
                centroid, metadata = generate_ngram_embedding_with_centroid(
                    text=text,
                    model=model,
                    min_n=min_n_gram,
                    max_n=max_n_gram
                )
                
                if centroid is not None:
                    # Zapisz centroid
                    save_work_embedding(
                        embedding=centroid,
                        openalex_id=openalex_id,
                        emb_type=emb_type,
                        model_name=model_name,
                        text=text,
                        output_dir=output_dir,
                        metadata=metadata
                    )
                    stats['generated'] += 1
                else:
                    stats['failed'] += 1
            
            except Exception as e:
                print(f"\n⚠️  Błąd dla {openalex_id}: {e}")
                stats['failed'] += 1
        
        return stats
    
    # Dla zwykłych embeddingów (title, abstract)
    to_process = []
    texts = []
    
    for idx, row in df_batch.iterrows():
        openalex_id = row['openalex_id']
        
        # Sprawdź czy już istnieje
        if skip_existing and embedding_exists(openalex_id, emb_type, output_dir):
            stats['skipped'] += 1
            continue
        
        to_process.append(row)
        texts.append(str(row[text_column]))
    
    # Jeśli nie ma nic do przetworzenia
    if not to_process:
        return stats
    
    try:
        # Generuj embeddingi dla całego batcha
        embeddings = model.encode(
            texts,
            batch_size=len(texts),
            show_progress_bar=False,
            convert_to_numpy=True
        )
        
        # Zapisz każdy embedding osobno
        for row, embedding, text in zip(to_process, embeddings, texts):
            try:
                save_work_embedding(
                    embedding=embedding,
                    openalex_id=row['openalex_id'],
                    emb_type=emb_type,
                    model_name=model_name,
                    text=text,
                    output_dir=output_dir
                )
                stats['generated'] += 1
            except Exception as e:
                print(f"\n⚠️  Błąd zapisu dla {row['openalex_id']}: {e}")
                stats['failed'] += 1
    
    except Exception as e:
        print(f"\n❌ Błąd generowania embeddingów dla batcha: {e}")
        stats['failed'] += len(to_process)
    
    return stats


print("✅ Funkcje zdefiniowane")

## 5. Generowanie embeddingów

Główna sekcja - generuje embeddingi dla wszystkich prac.

In [ ]:
print("="*80)
print("ROZPOCZĘCIE GENEROWANIA EMBEDDINGÓW")
print("="*80)
print(f"Modele: {MODELS}")
print(f"Typy: {EMB_TYPES}")
print(f"Prace do przetworzenia: {len(df_works_clean)}")
print(f"Batch size: {BATCH_SIZE}")
print()

# Statystyki - zagnieżdżone: model -> typ -> metric
total_stats = {}

# Główna pętla - iteruj przez modele
for model_idx, model_name in enumerate(MODELS, 1):
    print(f"\n{'='*80}")
    print(f"MODEL {model_idx}/{len(MODELS)}: {model_name}")
    print(f"{'='*80}")
    
    # Normalizacja nazwy modelu
    model_name_normalized = model_name.replace('/', '-').replace('\\', '-')
    
    # Inicjalizuj statystyki dla tego modelu
    total_stats[model_name] = {}
    for emb_type in EMB_TYPES:
        total_stats[model_name][emb_type] = {'generated': 0, 'skipped': 0, 'failed': 0}
    
    # Załaduj model (raz na model)
    print(f"⏳ Ładowanie modelu {model_name}...")
    model = SentenceTransformer(model_name)
    print(f"✅ Model załadowany")
    print(f"   Wymiar embeddingów: {model.get_sentence_embedding_dimension()}")
    print()
    
    # Pętla przez typy embeddingów
    for emb_idx, emb_type in enumerate(EMB_TYPES, 1):
        print(f"\n{'-'*80}")
        print(f"MODEL: {model_name} | TYP {emb_idx}/{len(EMB_TYPES)}: {emb_type.upper()}")
        print(f"{'-'*80}")
        
        # Ustal kolumnę i katalog w zależności od typu
        if emb_type == 'n_grams':
            text_column = 'abstract'  # N-gramy zawsze z abstractu
            output_dir = os.path.join(EMBEDDINGS_BASE_DIR, 'n_grams', model_name_normalized)
            print(f"📝 N-gramy z abstractów (min={MIN_N_GRAM}, max={MAX_N_GRAM or 'auto'})")
        elif emb_type == 'title':
            text_column = 'title'
            output_dir = os.path.join(EMBEDDINGS_BASE_DIR, 'titles', model_name_normalized)
        elif emb_type == 'abstract':
            text_column = 'abstract'
            output_dir = os.path.join(EMBEDDINGS_BASE_DIR, 'abstracts', model_name_normalized)
        else:
            print(f"⚠️ Nieznany typ: {emb_type}, pomijam")
            continue
        
        # Utwórz katalog jeśli nie istnieje
        os.makedirs(output_dir, exist_ok=True)
        print(f"📁 Katalog: {output_dir}")
        
        # Przetwarzanie w batchach z progress bar
        with tqdm(total=len(df_works_clean), desc=f"{model_name_normalized}/{emb_type}") as pbar:
            for i in range(0, len(df_works_clean), BATCH_SIZE):
                batch = df_works_clean.iloc[i:i+BATCH_SIZE]
                
                stats = generate_embeddings_batch(
                    df_batch=batch,
                    model=model,
                    text_column=text_column,
                    emb_type=emb_type,
                    output_dir=output_dir,
                    model_name=model_name,
                    skip_existing=SKIP_EXISTING,
                    min_n_gram=MIN_N_GRAM,
                    max_n_gram=MAX_N_GRAM
                )
                
                # Aktualizuj statystyki
                for key in stats:
                    total_stats[model_name][emb_type][key] += stats[key]
                
                # Aktualizuj progress bar
                pbar.update(len(batch))
        
        print(f"✅ Zakończono dla {model_name}/{emb_type}:")
        print(f"   Wygenerowano: {total_stats[model_name][emb_type]['generated']}")
        print(f"   Pominięto: {total_stats[model_name][emb_type]['skipped']}")
        print(f"   Błędy: {total_stats[model_name][emb_type]['failed']}")

print(f"\n\n{'='*80}")
print("GENEROWANIE ZAKOŃCZONE")
print(f"{'='*80}")
print(f"\n📊 PODSUMOWANIE KOŃCOWE:\n")

# Wyświetl statystyki dla każdego modelu
for model_name in MODELS:
    print(f"\n🤖 {model_name}:")
    for emb_type in EMB_TYPES:
        stats = total_stats[model_name][emb_type]
        print(f"   {emb_type.capitalize()}:")
        print(f"      Wygenerowano: {stats['generated']}")
        print(f"      Pominięto: {stats['skipped']}")
        print(f"      Błędy: {stats['failed']}")

# Łączne statystyki
total_generated = sum(
    total_stats[m][t]['generated'] 
    for m in MODELS 
    for t in EMB_TYPES
)
total_skipped = sum(
    total_stats[m][t]['skipped'] 
    for m in MODELS 
    for t in EMB_TYPES
)
total_failed = sum(
    total_stats[m][t]['failed'] 
    for m in MODELS 
    for t in EMB_TYPES
)

print(f"\n{'='*80}")
print(f"📈 ŁĄCZNIE:")
print(f"   Wygenerowano: {total_generated} embeddingów")
print(f"   Pominięto: {total_skipped}")
print(f"   Błędy: {total_failed}")
print(f"{'='*80}")

## 6. Weryfikacja

Sprawdź czy wszystko się wygenerowało poprawnie.

In [ ]:
print("🔍 Weryfikacja wygenerowanych embeddingów...\n")

# Sprawdź wszystkie kombinacje modeli i typów
for model_name in MODELS:
    model_name_normalized = model_name.replace('/', '-').replace('\\', '-')
    
    print(f"\n{'='*60}")
    print(f"🤖 {model_name}")
    print(f"{'='*60}")
    
    for emb_type in EMB_TYPES:
        # Ustal ścieżkę w zależności od typu
        if emb_type == 'n_grams':
            subdir = 'n_grams'
        elif emb_type == 'title':
            subdir = 'titles'
        else:  # abstract
            subdir = 'abstracts'
        
        emb_dir = os.path.join(EMBEDDINGS_BASE_DIR, subdir, model_name_normalized)
        
        # Zlicz pliki
        if os.path.exists(emb_dir):
            files = list(Path(emb_dir).glob('*.pkl'))
            print(f"   📁 {emb_type.capitalize()}: {len(files)} plików")
            
            # Sprawdź przykładowy plik
            if files:
                sample_file = files[0]
                with open(sample_file, 'rb') as f:
                    sample_data = pickle.load(f)
                
                print(f"      └─ Przykład: {sample_file.name}")
                print(f"         Model: {sample_data['model']}")
                print(f"         Wymiar: {sample_data['embedding_dim']}")
                
                # Dla N-gramów pokaż metadane
                if emb_type == 'n_grams' and 'metadata' in sample_data:
                    meta = sample_data['metadata']
                    print(f"         Zdań: {meta.get('num_sentences', 'N/A')}")
                    print(f"         N-gramów: {meta.get('num_ngrams', 'N/A')}")
                    print(f"         Długości N: {meta.get('ngram_lengths', 'N/A')}")
                
                print(f"         Tekst (100 znaków): {sample_data['text'][:100]}...")
        else:
            print(f"   ⚠️  {emb_type.capitalize()}: katalog nie istnieje")

print(f"\n{'='*60}")
print("✅ Weryfikacja zakończona")

## 7. Demonstracja N-gramów

Przykład działania N-gramów na przykładowym tekście:

In [ ]:
# Przykładowy abstract
example_text = """
Machine learning has revolutionized artificial intelligence. 
Deep neural networks achieve state-of-the-art results in many tasks. 
Transfer learning enables faster model training. 
Pre-trained models save computational resources. 
Natural language processing benefits greatly from these advances.
"""

print("📝 PRZYKŁADOWY TEKST:")
print(example_text.strip())
print()

# Podziel na zdania
sentences = split_into_sentences(example_text)
print(f"\n🔍 PODZIAŁ NA ZDANIA:")
print(f"Liczba zdań: {len(sentences)}")
for i, sent in enumerate(sentences, 1):
    print(f"   S{i}: {sent}")

# Utwórz N-gramy z ograniczeniem do 3
ngrams_limited = create_sentence_ngrams(sentences, min_n=1, max_n=3)
print(f"\n📊 N-GRAMY (min=1, max=3):")
print(f"Liczba N-gramów: {len(ngrams_limited)}")
print()

current_n = 1
for i, ngram in enumerate(ngrams_limited):
    # Oblicz długość N-gramu (ile zdań)
    ngram_len = sum(1 for s in sentences if s in ngram)
    if ngram_len > current_n:
        current_n = ngram_len
        print()
    
    print(f"   {current_n}-gram #{i+1}: {ngram[:80]}{'...' if len(ngram) > 80 else ''}")

# Utwórz wszystkie możliwe N-gramy
ngrams_all = create_sentence_ngrams(sentences, min_n=1, max_n=None)
print(f"\n📊 WSZYSTKIE N-GRAMY (min=1, max=auto):")
print(f"Liczba N-gramów: {len(ngrams_all)}")
print()

# Oblicz oczekiwaną liczbę N-gramów
n_sents = len(sentences)
expected = sum(n_sents - i for i in range(n_sents))
print(f"💡 Dla {n_sents} zdań:")
print(f"   1-gramów: {n_sents}")
print(f"   2-gramów: {n_sents - 1}")
print(f"   3-gramów: {n_sents - 2}")
print(f"   ...")
print(f"   {n_sents}-gramów: 1")
print(f"   SUMA: {expected} N-gramów")
print()
print(f"✅ Embedding będzie średnią z {expected} różnych embeddingów!")

In [ ]:
# Przykład: wczytaj embedding dla konkretnej pracy
example_id = df_works_clean.iloc[0]['openalex_id']

print(f"📖 Przykład wczytania embeddingów dla: {example_id}\n")

# Użyj pierwszego modelu z listy jako przykład
example_model = MODELS[0]
model_name_normalized = example_model.replace('/', '-').replace('\\', '-')

print(f"🤖 Model: {example_model}\n")

# Sprawdź wszystkie dostępne typy
for emb_type in EMB_TYPES:
    # Ustal ścieżkę
    if emb_type == 'n_grams':
        subdir = 'n_grams'
    elif emb_type == 'title':
        subdir = 'titles'
    else:  # abstract
        subdir = 'abstracts'
    
    emb_dir = os.path.join(EMBEDDINGS_BASE_DIR, subdir, model_name_normalized)
    
    # Wczytaj embedding
    emb_data = load_work_embedding(example_id, emb_type, emb_dir)
    
    if emb_data:
        print(f"✅ {emb_type.capitalize()} embedding:")
        print(f"   Model: {emb_data['model']}")
        print(f"   Wymiar: {emb_data['embedding_dim']}")
        
        # Dla N-gramów pokaż metadane
        if emb_type == 'n_grams' and 'metadata' in emb_data:
            meta = emb_data['metadata']
            print(f"   📊 Statystyki N-gramów:")
            print(f"      Zdań w abstracie: {meta.get('num_sentences', 'N/A')}")
            print(f"      Wygenerowanych N-gramów: {meta.get('num_ngrams', 'N/A')}")
            print(f"      Długości N-gramów: {meta.get('ngram_lengths', 'N/A')}")
            print(f"      Min N: {meta.get('min_n', 'N/A')}, Max N: {meta.get('max_n', 'N/A')}")
        
        print(f"   Tekst (200 znaków): {emb_data['text'][:200]}...")
        print(f"   Embedding (pierwsze 5 wartości): {emb_data['embedding'][:5]}")
        print()
    else:
        print(f"❌ Nie znaleziono {emb_type} embedingu dla {example_id}")
        print()

## 8. Podsumowanie

### Konfiguracja:

Notebook obsługuje **generowanie embeddingów dla wielu modeli i typów jednocześnie**:

```python
# Lista modeli - dodaj dowolne modele
MODELS = [
    'all-MiniLM-L6-v2',
    'allenai/specter',
    'all-mpnet-base-v2',
]

# Typy embeddingów - wybierz co chcesz generować
EMB_TYPES = [
    'title',      # Embedding tytułu
    'abstract',   # Embedding całego abstractu
    'n_grams',    # Embedding centrum N-gramów zdań z abstractu
]

# Konfiguracja N-gramów (tylko dla typu 'n_grams')
MIN_N_GRAM = 1         # Minimalna długość N-gramu (pojedyncze zdania)
MAX_N_GRAM = None      # Maksymalna długość (None = cały abstract)
```

Notebook automatycznie:
- Iteruje przez wszystkie modele
- Dla każdego modelu generuje embeddingi dla wybranych typów
- Zapisuje w odpowiednich podfolderach
- Wyświetla postęp i statystyki

### Co to są N-gramy zdań?

Typ **'n_grams'** działa inaczej niż 'title' i 'abstract':

1. **Dzieli abstract na zdania**
   - Przykład: 5 zdań → [S1, S2, S3, S4, S5]

2. **Tworzy N-gramy różnych długości**
   - 1-gramy: [S1], [S2], [S3], [S4], [S5]
   - 2-gramy: [S1+S2], [S2+S3], [S3+S4], [S4+S5]
   - 3-gramy: [S1+S2+S3], [S2+S3+S4], [S3+S4+S5]
   - ...
   - N-gram: [S1+S2+S3+S4+S5] (cały abstract)

3. **Embeduje każdy N-gram osobno**
   - Dla 5 zdań: 5 + 4 + 3 + 2 + 1 = 15 embeddingów

4. **Oblicza centrum (centroid)**
   - Średnia wszystkich 15 embeddingów → **jeden finalny embedding**

**Zalety N-gramów:**
- Uchwycają różne konteksty (lokalne i globalne)
- Bardziej odporne na szum niż pojedyncze zdania
- Uwzględniają strukturę hierarchiczną tekstu

### Struktura plików:

```
embeddings/
├── titles/
│   ├── all-MiniLM-L6-v2/
│   │   └── W1234567890_title.pkl
│   └── allenai-specter/
│       └── W1234567890_title.pkl
├── abstracts/
│   ├── all-MiniLM-L6-v2/
│   │   └── W1234567890_abstract.pkl
│   └── allenai-specter/
│       └── W1234567890_abstract.pkl
└── n_grams/
    ├── all-MiniLM-L6-v2/
    │   └── W1234567890_n_grams.pkl
    └── allenai-specter/
        └── W1234567890_n_grams.pkl
```

**Każdy model ma swój osobny podfolder!**

### Format pliku:

Każdy plik .pkl zawiera dict:
```python
{
    'embedding': np.ndarray,      # Wygenerowany embedding (lub centroid dla n_grams)
    'openalex_id': str,            # ID pracy
    'type': str,                   # 'title', 'abstract' lub 'n_grams'
    'model': str,                  # Nazwa użytego modelu
    'text': str,                   # Oryginalny tekst (dla referencji)
    'embedding_dim': int,          # Wymiar embedingu
    'metadata': dict               # Opcjonalne (tylko dla n_grams)
}
```

**Metadane dla N-gramów:**
```python
'metadata': {
    'num_sentences': 5,           # Liczba zdań w abstracie
    'num_ngrams': 15,             # Liczba wygenerowanych N-gramów
    'ngram_lengths': [1,2,3,4,5], # Długości użytych N-gramów
    'min_n': 1,                   # Min długość N-gramu
    'max_n': None                 # Max długość N-gramu
}
```

### Przykłady użycia:

**1. Tylko N-gramy dla jednego modelu:**
```python
MODELS = ['all-MiniLM-L6-v2']
EMB_TYPES = ['n_grams']
MIN_N_GRAM = 1
MAX_N_GRAM = None  # Wszystkie długości
```

**2. Title + N-gramy:**
```python
MODELS = ['allenai/specter']
EMB_TYPES = ['title', 'n_grams']
```

**3. Wszystkie typy dla wielu modeli:**
```python
MODELS = [
    'all-MiniLM-L6-v2',
    'allenai/specter',
    'all-mpnet-base-v2',
]
EMB_TYPES = ['title', 'abstract', 'n_grams']
# To wygeneruje 3 × 3 = 9 kombinacji
```

**4. N-gramy z ograniczoną długością:**
```python
EMB_TYPES = ['n_grams']
MIN_N_GRAM = 1
MAX_N_GRAM = 3  # Tylko 1-gramy, 2-gramy i 3-gramy
# Dla 10 zdań: 10 + 9 + 8 = 27 N-gramów zamiast 55
```

### Wznowienie:

Jeśli proces się przerwie, po prostu uruchom ponownie komórkę generowania.
Dzięki `SKIP_EXISTING = True` pominie już wygenerowane embeddingi.

**Uwaga:** Pomijanie działa na poziomie poszczególnych plików, więc:
- Jeśli przerwiesz w trakcie generowania dla modelu A, pozostałe modele nie zostaną pomięte
- Każdy plik jest sprawdzany osobno, więc wznowienie jest bezpieczne

### Dodawanie nowych modeli lub typów:

Aby dodać nowe modele lub typy później:
1. Dodaj nazwę modelu do listy `MODELS` lub typ do `EMB_TYPES`
2. Uruchom ponownie notebook
3. Nowe kombinacje zostaną wygenerowane, istniejące pominięte

### Dlaczego prace są pomijane?

Prace mogą być pominięte z trzech powodów:

1. **SKIP_EXISTING = True** - plik już istnieje (wznowienie po przerwie)
   - Sprawdź katalog - jeśli tam są pliki, to dlatego
   
2. **Brak danych** - praca nie ma wypełnionego title lub abstract
   - Filtrujemy: `df.dropna(subset=['openalex_id', 'title', 'abstract'])`
   - Sprawdź w sekcji "Wczytanie danych" ile prac zostało po filtracji
   
3. **Błędy** - coś poszło nie tak przy generowaniu
   - Zobacz kolumnę "Błędy" w statystykach końcowych
   - Dla N-gramów: błąd może wystąpić jeśli abstract jest zbyt krótki lub nie da się podzielić na zdania

### Czas wykonania:

Czas zależy od:
- Liczby modeli × liczby typów
- Liczby prac do przetworzenia
- Rozmiaru batcha
- Mocy obliczeniowej (GPU vs CPU)

**Uwaga dla N-gramów:** 
- N-gramy są **wolniejsze** niż zwykłe embeddingi
- Dla abstractu z 10 zdaniami: 55 embeddingów zamiast 1
- Czas może być 10-50x dłuższy niż dla zwykłego abstractu
- Można ograniczyć `MAX_N_GRAM` aby przyspieszyć

**Przykład:** 
- 3 modele × ['title', 'abstract'] × 3000 prac ≈ 30-60 minut (CPU)
- 3 modele × ['title', 'abstract', 'n_grams'] × 3000 prac ≈ 2-4 godziny (CPU)